# Phát hiện tin giả tiếng Việt với SVM trên bộ dữ liệu VFND

Notebook này sử dụng bộ dữ liệu **VFND - Vietnamese Fake News Dataset** từ GitHub:

- Repository: `WhySchools/VFND-vietnamese-fake-news-datasets`
- File chính dùng trong notebook: `CSV/vn_news_226_tlfr.csv`
- Dữ liệu gồm 2 cột chính:
  - `text`: nội dung tin tức
  - `label`: nhãn phân loại
- Quy ước nhãn:
  - `0`: Real - tin thật
  - `1`: Fake - tin giả

## Mục tiêu

1. Tải hoặc đọc dataset VFND.
2. Tiền xử lý văn bản tiếng Việt.
3. Biểu diễn văn bản bằng TF-IDF.
4. Huấn luyện mô hình **SVM tuyến tính** để phân loại tin giả / tin thật.
5. Tối ưu tham số bằng `GridSearchCV`.
6. Đánh giá mô hình bằng Accuracy, Precision, Recall, F1-score và Confusion Matrix.
7. So sánh SVM với Naive Bayes và Logistic Regression.
8. Lưu mô hình tốt nhất để dùng lại.

## 1. Cài đặt và import thư viện

Nếu thiếu thư viện, bạn có thể chạy ô dưới đây:

```python
!pip install pandas numpy scikit-learn matplotlib seaborn joblib
```

In [ ]:
# Nếu máy bạn chưa có thư viện, bỏ dấu # ở dòng dưới rồi chạy
# !pip install pandas numpy scikit-learn matplotlib seaborn joblib

import os
import re
import csv
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from io import StringIO
from urllib.request import urlopen

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

RANDOM_STATE = 42

plt.rcParams["figure.figsize"] = (8, 5)
sns.set_theme(style="whitegrid")

## 2. Khai báo đường dẫn dataset

Notebook mặc định tải file CSV trực tiếp từ GitHub.

Nếu bạn muốn chạy offline, hãy tải file `vn_news_226_tlfr.csv` về cùng thư mục với notebook rồi đổi:

```python
DATA_SOURCE = "vn_news_226_tlfr.csv"
```

In [ ]:
RAW_URL_226 = "https://raw.githubusercontent.com/WhySchools/VFND-vietnamese-fake-news-datasets/master/CSV/vn_news_226_tlfr.csv"
RAW_URL_223 = "https://raw.githubusercontent.com/WhySchools/VFND-vietnamese-fake-news-datasets/master/CSV/vn_news_223_tdlfr.csv"

# Dùng file 226 bản ghi: text, label
DATA_SOURCE = RAW_URL_226

# Nếu chạy offline, đổi thành:
# DATA_SOURCE = "vn_news_226_tlfr.csv"

## 3. Hàm đọc dữ liệu VFND

Một số file CSV chứa văn bản dài, trong văn bản có dấu xuống dòng. Vì vậy, ta dùng hàm đọc dữ liệu linh hoạt hơn thay vì chỉ gọi `pd.read_csv()` đơn giản.

In [ ]:
def load_vfnd_csv(source):
    """
    Đọc file CSV của VFND.

    Tham số:
        source: đường dẫn local hoặc URL raw GitHub.

    Trả về:
        DataFrame có ít nhất 2 cột: text, label.
    """
    # Cách 1: thử đọc trực tiếp bằng pandas
    try:
        df = pd.read_csv(source, encoding="utf-8", engine="python")
        df.columns = [str(c).strip().lower() for c in df.columns]

        if "text" in df.columns and "label" in df.columns:
            return df
    except Exception as e:
        print("Không đọc được trực tiếp bằng pandas, chuyển sang cách đọc thủ công.")
        print("Lỗi:", e)

    # Cách 2: đọc thủ công nội dung CSV
    if str(source).startswith("http"):
        raw_text = urlopen(source).read().decode("utf-8", errors="replace")
    else:
        with open(source, "r", encoding="utf-8", errors="replace") as f:
            raw_text = f.read()

    csv_reader = csv.reader(
        StringIO(raw_text),
        delimiter=",",
        quotechar='"',
        skipinitialspace=True
    )

    rows = list(csv_reader)

    # Bỏ các dòng rỗng
    rows = [row for row in rows if len(row) > 0]

    # Nếu dòng đầu là header
    header = [x.strip().lower() for x in rows[0]]

    if "text" in header and "label" in header:
        text_idx = header.index("text")
        label_idx = header.index("label")
        data_rows = rows[1:]
    else:
        # Với trường hợp file bị lỗi header, giả định:
        # cột đầu là text, cột cuối là label
        text_idx = 0
        label_idx = -1
        data_rows = rows

    data = []
    for row in data_rows:
        if len(row) < 2:
            continue

        text = row[text_idx].strip()
        label = row[label_idx].strip()

        if text and label != "":
            data.append({
                "text": text,
                "label": label
            })

    df = pd.DataFrame(data)
    return df


df = load_vfnd_csv(DATA_SOURCE)

print("Kích thước dữ liệu:", df.shape)
df.head()

## 4. Kiểm tra dữ liệu ban đầu

In [ ]:
print("Thông tin dữ liệu:")
display(df.info())

print("\n5 dòng đầu:")
display(df.head())

print("\nTên cột:")
print(df.columns.tolist())

print("\nSố lượng giá trị thiếu:")
print(df.isna().sum())

print("\nPhân bố nhãn ban đầu:")
print(df["label"].value_counts(dropna=False))

## 5. Chuẩn hóa nhãn

Theo mô tả của dataset VFND:

- `0`: tin thật - Real
- `1`: tin giả - Fake

In [ ]:
def normalize_label(label):
    """
    Chuẩn hóa nhãn về dạng số:
        0: Real
        1: Fake
    """
    label = str(label).strip().lower()

    if label in ["0", "real", "true", "thật", "tin thật"]:
        return 0
    if label in ["1", "fake", "false", "giả", "tin giả"]:
        return 1

    # Nếu label có dạng "0.0", "1.0"
    try:
        value = int(float(label))
        if value in [0, 1]:
            return value
    except:
        pass

    return np.nan


df = df[["text", "label"]].copy()
df["label"] = df["label"].apply(normalize_label)

df = df.dropna(subset=["text", "label"])
df["label"] = df["label"].astype(int)

label_names = {
    0: "Real - Tin thật",
    1: "Fake - Tin giả"
}

print("Kích thước sau khi chuẩn hóa:", df.shape)
print(df["label"].value_counts().rename(index=label_names))

## 6. Tiền xử lý văn bản tiếng Việt

Các bước xử lý:

- Chuyển về chữ thường.
- Xóa URL.
- Xóa email.
- Xóa số.
- Xóa ký tự đặc biệt.
- Xóa khoảng trắng thừa.
- Loại bỏ một số stopword tiếng Việt cơ bản.

Lưu ý: Với bài nâng cao, có thể dùng `underthesea` hoặc `pyvi` để tách từ tiếng Việt. Ở notebook này, ta dùng cách đơn giản để dễ chạy trên mọi máy.

In [ ]:
vietnamese_stopwords = set([
    "và", "là", "của", "có", "cho", "với", "một", "các", "những",
    "được", "trong", "khi", "đã", "này", "đó", "thì", "mà", "ở",
    "từ", "về", "theo", "sau", "trước", "đến", "ra", "vào", "nên",
    "như", "trên", "dưới", "bị", "cũng", "nhiều", "rất", "lại",
    "sẽ", "đang", "không", "người", "việc", "năm", "ngày"
])

def clean_text(text):
    text = str(text).lower()

    # Xóa URL
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Xóa email
    text = re.sub(r"\S+@\S+", " ", text)

    # Xóa số
    text = re.sub(r"\d+", " ", text)

    # Giữ lại chữ cái, dấu tiếng Việt và khoảng trắng
    text = re.sub(r"[^a-zA-ZÀ-ỹà-ỹ\s]", " ", text)

    # Xóa khoảng trắng thừa
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [word for word in tokens if word not in vietnamese_stopwords]

    return " ".join(tokens)


df["clean_text"] = df["text"].apply(clean_text)

# Loại bỏ văn bản rỗng và trùng lặp
df = df[df["clean_text"].str.len() > 0].copy()
df = df.drop_duplicates(subset=["clean_text"]).reset_index(drop=True)

print("Kích thước sau tiền xử lý:", df.shape)
display(df[["text", "clean_text", "label"]].head())

## 7. Trực quan hóa phân bố nhãn

In [ ]:
label_count = df["label"].value_counts().sort_index()
print(label_count.rename(index=label_names))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="label")
plt.title("Phân bố nhãn Real/Fake trong VFND")
plt.xlabel("Nhãn")
plt.ylabel("Số lượng")
plt.xticks([0, 1], ["Real", "Fake"])
plt.show()

## 8. Chia dữ liệu train/test

Ta chia dữ liệu thành:

- 80% train
- 20% test

Dùng `stratify=y` để giữ tỷ lệ nhãn giữa train và test.

In [ ]:
X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Số mẫu train:", len(X_train))
print("Số mẫu test:", len(X_test))

print("\nPhân bố nhãn train:")
print(y_train.value_counts().sort_index().rename(index=label_names))

print("\nPhân bố nhãn test:")
print(y_test.value_counts().sort_index().rename(index=label_names))

## 9. Xây dựng mô hình SVM với TF-IDF

Pipeline gồm 2 bước:

1. `TfidfVectorizer`: chuyển văn bản thành vector TF-IDF.
2. `LinearSVC`: mô hình SVM tuyến tính.

SVM tuyến tính thường phù hợp với bài toán phân loại văn bản vì dữ liệu TF-IDF có số chiều lớn và thưa.

In [ ]:
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("svm", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

svm_pipeline.fit(X_train, y_train)

y_pred_svm = svm_pipeline.predict(X_test)

print("Kết quả mô hình SVM ban đầu:")
print(classification_report(
    y_test,
    y_pred_svm,
    target_names=["Real - Tin thật", "Fake - Tin giả"],
    zero_division=0
))

## 10. Tối ưu mô hình SVM bằng GridSearchCV

Ta thử nhiều giá trị tham số:

- `tfidf__max_features`: số lượng đặc trưng tối đa.
- `tfidf__ngram_range`: dùng unigram hoặc unigram + bigram.
- `svm__C`: tham số điều chuẩn của SVM.

Vì dataset nhỏ, GridSearchCV có thể chạy khá nhanh.

In [ ]:
param_grid = {
    "tfidf__max_features": [5000, 10000, 20000],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "svm__C": [0.1, 1, 10]
}

grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Tham số tốt nhất:")
print(grid_search.best_params_)

print("\nF1-score trung bình tốt nhất trên cross-validation:")
print(grid_search.best_score_)

best_svm_model = grid_search.best_estimator_

## 11. Đánh giá SVM sau tối ưu trên tập test

In [ ]:
y_pred_best_svm = best_svm_model.predict(X_test)

svm_accuracy = accuracy_score(y_test, y_pred_best_svm)
svm_precision = precision_score(y_test, y_pred_best_svm, zero_division=0)
svm_recall = recall_score(y_test, y_pred_best_svm, zero_division=0)
svm_f1 = f1_score(y_test, y_pred_best_svm, zero_division=0)

print("Accuracy:", svm_accuracy)
print("Precision:", svm_precision)
print("Recall:", svm_recall)
print("F1-score:", svm_f1)

print("\nBáo cáo phân loại chi tiết:")
print(classification_report(
    y_test,
    y_pred_best_svm,
    target_names=["Real - Tin thật", "Fake - Tin giả"],
    zero_division=0
))

## 12. Ma trận nhầm lẫn của SVM

In [ ]:
cm = confusion_matrix(y_test, y_pred_best_svm)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Real", "Fake"],
    yticklabels=["Real", "Fake"]
)
plt.title("Confusion Matrix - SVM")
plt.xlabel("Nhãn dự đoán")
plt.ylabel("Nhãn thực tế")
plt.show()

## 13. So sánh SVM với các thuật toán khác

Ta so sánh SVM với:

- Multinomial Naive Bayes
- Logistic Regression

Các mô hình đều dùng TF-IDF để biểu diễn văn bản.

In [ ]:
models = {
    "SVM tuyến tính": best_svm_model,

    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=20000,
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True
        )),
        ("nb", MultinomialNB())
    ]),

    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=20000,
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True
        )),
        ("lr", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])
}

results = []

for model_name, model in models.items():
    print("=" * 70)
    print("Đang huấn luyện/đánh giá:", model_name)

    # best_svm_model đã train rồi, nhưng fit lại để thống nhất quy trình
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    result = {
        "Mô hình": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0)
    }

    results.append(result)

    print(classification_report(
        y_test,
        y_pred,
        target_names=["Real - Tin thật", "Fake - Tin giả"],
        zero_division=0
    ))

results_df = pd.DataFrame(results).sort_values(by="F1-score", ascending=False)
display(results_df)

## 14. Biểu đồ so sánh các mô hình

In [ ]:
results_melted = results_df.melt(
    id_vars="Mô hình",
    value_vars=["Accuracy", "Precision", "Recall", "F1-score"],
    var_name="Chỉ số",
    value_name="Giá trị"
)

plt.figure(figsize=(10, 6))
sns.barplot(data=results_melted, x="Mô hình", y="Giá trị", hue="Chỉ số")
plt.title("So sánh SVM với các mô hình khác trên VFND")
plt.ylim(0, 1.05)
plt.xticks(rotation=15)
plt.legend(loc="lower right")
plt.show()

## 15. Cross-validation cho SVM

Ngoài tập test, ta kiểm tra thêm bằng cross-validation trên toàn bộ dữ liệu để có cái nhìn ổn định hơn.

In [ ]:
cv_scores = cross_val_score(
    best_svm_model,
    X,
    y,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

print("F1-score từng fold:", cv_scores)
print("F1-score trung bình:", cv_scores.mean())
print("Độ lệch chuẩn:", cv_scores.std())

## 16. Thử dự đoán tin mới

Bạn có thể thay nội dung trong danh sách `sample_news` bằng tin khác để kiểm tra.

In [ ]:
def predict_fake_news(text, model=best_svm_model):
    cleaned = clean_text(text)
    pred = model.predict([cleaned])[0]
    return label_names[pred]


sample_news = [
    "Bộ Y tế khuyến cáo người dân đeo khẩu trang tại nơi đông người để phòng bệnh.",
    "Một loại thuốc bí mật có thể chữa khỏi mọi bệnh chỉ sau một ngày.",
    "Ngân hàng Nhà nước công bố điều chỉnh lãi suất nhằm ổn định thị trường.",
    "Người ngoài hành tinh xuất hiện tại Hà Nội và bắt tay với lãnh đạo."
]

for news in sample_news:
    print("Tin:", news)
    print("Dự đoán:", predict_fake_news(news))
    print("-" * 80)

## 17. Lưu mô hình

Mô hình sau khi lưu có thể dùng lại mà không cần huấn luyện từ đầu.

In [ ]:
MODEL_PATH = "svm_vfnd_fake_news_model.pkl"

joblib.dump({
    "model": best_svm_model,
    "label_names": label_names,
    "clean_text_function_note": "Cần dùng lại hàm clean_text tương tự khi deploy."
}, MODEL_PATH)

print("Đã lưu mô hình vào:", MODEL_PATH)

## 18. Tải lại mô hình và dự đoán

Ô này minh họa cách load mô hình đã lưu.

In [ ]:
loaded = joblib.load(MODEL_PATH)
loaded_model = loaded["model"]

test_text = "Một thông tin chưa kiểm chứng lan truyền trên mạng xã hội khiến nhiều người hoang mang."
test_cleaned = clean_text(test_text)

pred = loaded_model.predict([test_cleaned])[0]

print("Tin:", test_text)
print("Dự đoán:", label_names[pred])

## 19. Kết luận mẫu cho báo cáo

Trong bài toán phân loại tin giả tiếng Việt, nhóm sử dụng bộ dữ liệu VFND gồm các văn bản tin tức được gán nhãn `Real` và `Fake`. Dữ liệu được tiền xử lý bằng các bước như chuyển chữ thường, loại bỏ URL, ký tự đặc biệt, số và stopword cơ bản. Sau đó, văn bản được biểu diễn bằng phương pháp TF-IDF.

Mô hình chính được sử dụng là SVM tuyến tính thông qua `LinearSVC`. Đây là lựa chọn phù hợp cho bài toán phân loại văn bản vì dữ liệu văn bản sau khi biến đổi TF-IDF thường có số chiều lớn và thưa. Nhóm cũng thực hiện tối ưu tham số bằng `GridSearchCV` để lựa chọn cấu hình tốt hơn cho mô hình SVM.

Kết quả được đánh giá bằng các chỉ số Accuracy, Precision, Recall và F1-score. Ngoài ra, nhóm so sánh SVM với Naive Bayes và Logistic Regression để có cơ sở nhận xét. Nếu SVM đạt F1-score cao nhất hoặc gần cao nhất, có thể kết luận rằng SVM là một mô hình phù hợp cho bài toán phát hiện tin giả tiếng Việt trên bộ dữ liệu VFND.